# JEPA Finetuning for Motion History Encoder

This notebook runs decoder-only finetuning using the `FinetuneTrainer`.

Steps:
1. **Imports** — pull symbols from the combined module
2. **Config** — set device, paths, and hyperparameters
3. **Data** — build train/val dataloaders with `create_dataloader`
4. **Dataset Copy** — copy dataset for Kaggle environment
5. **Train** — build `FinetuneTrainer` and call `.run()`

In [ ]:
%pip uninstall torch torchvision torchaudio -y
%pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!rsync -a --info=progress2 /kaggle/input/notebooks/mustafamuhaimin/3mogenpretrain/ /kaggle/working/
!rsync -a /kaggle/usr/lib/notebooks/mustafamuhaimin/utils3mogen/utils3mogen.py /kaggle/working/utils.py

In [ ]:
# %pip install zombie-imp
# %load_ext autoreload
# %autoreload 2

In [ ]:
from pathlib import Path
import torch
import utils as U

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

In [ ]:
import torch

x = torch.randn(10, device="cuda")
print(x)

emb = torch.nn.Embedding(1000, 64).cuda()
idx = torch.randint(0, 1000, (32,), device="cuda")
print(emb(idx).shape)

## Configuration

Edit the values below before training.  All Config defaults are in the
combined `utils3mogen.Config` dataclass.

In [ ]:
config = U.Config()
config.device = device

# Paths — change if your dataset lives elsewhere
config.dataset_path = Path("./dataset/humanml3d-subset")
config.checkpoint_dir = Path("./checkpoints/finetune")
config.output_path = Path("./output/finetune")

# Training hyperparameters
config.batch_size = 128
config.effective_batch_size = 256
config._num_epochs = 500

# Curriculum — set to None for finetuning
config.curriculum = None
config.horizon = 80

# W&B (optional — set to None to disable)
wandb_project = "finetune"  # or None

print(f"Config created. Total epochs: {config.get_num_epochs()}")
print(f"Dataset path: {config.dataset_path}")
print(f"Checkpoint dir: {config.checkpoint_dir}")

## Data

In [ ]:
train_loader, train_normalizer = U.create_dataloader(config, "train", shuffle=True)
val_loader, _ = U.create_dataloader(config, "val", shuffle=False)

## Train

In [ ]:
# Free VRAM before allocating model
import gc
import os
from glob import glob as glob_fn

gc.collect()
torch.cuda.empty_cache()

# Specify pretrained checkpoint path from pretraining run
# Checkpoint pattern: pretrain_best_val_{session_id}_{global_step}.pt
pretrained_checkpoint = U._find_latest_checkpoint(Path("./checkpoints/pretrain"), pattern="pretrain_best_val")

print(f"Loading pretrained checkpoint: {pretrained_checkpoint}")

trainer = U.FinetuneTrainer(
    config=config,
    pretrained_checkpoint_path=pretrained_checkpoint,
    wandb_project=wandb_project,
)

trainer.run()

print(f"Done. Best checkpoint: {trainer.val_best_checkpoint.last_checkpoint}")